# Importation des librairies

In [2]:
import requests # Cette librairie permet de faire des requêtes HTTP pour récupérer le texte du roman depuis un site web
import re # Cette librairie permet de faire du traitement de texte, notamment pour nettoyer le texte du roman
import pandas as pd # Cette librairie permet de manipuler des données sous forme de tableaux (DataFrames) et de les analyser
import numpy as np # Cette librairie permet de faire des calculs numériques et de manipuler des tableaux multidimensionnels 
import matplotlib.pyplot as plt # Cette librairie permet de faire des visualisations graphiques, notamment pour afficher des histogrammes, des nuages de mots, etc.
import seaborn as sns # Cette librairie permet de faire des visualisations graphiques plus avancées et est souvent utilisée en complément de matplotlib
from collections import Counter # Cette librairie permet de compter le nombre d'occurrences de chaque mot dans le texte du roman
from wordcloud import WordCloud # Cette librairie permet de générer des nuages de mots à partir du texte du roman, en mettant en avant les mots les plus fréquents

import nltk # Cette librairie permet de faire du traitement de texte, notamment pour tokeniser le texte en mots ou en phrases, supprimer les stopwords, faire de la lemmatisation, etc.
import spacy # Cette librairie permet de faire du traitement de texte plus avancé, notamment pour faire de la reconnaissance d'entités nommées, de la dépendance syntaxique, etc.
import sklearn # Cette librairie permet de faire de l'apprentissage automatique, notamment pour faire de la classification, de la régression, du clustering, etc.
from nltk.tokenize import word_tokenize, sent_tokenize # Ces fonctions permettent de tokeniser le texte du roman en mots ou en phrases
from nltk.corpus import stopwords # Cette fonction permet de récupérer la liste des stopwords en français, c'est-à-dire les mots qui sont très fréquents et qui n'apportent pas beaucoup d'information (comme "le", "la", "et", etc.)
from nltk.stem import SnowballStemmer # Cette classe permet de faire de la racinisation (stemming) en français, c'est-à-dire de réduire les mots à leur racine (par exemple, "manger", "mange", "mangé" seront tous réduits à "mang")
from nltk.stem import WordNetLemmatizer # Cette classe permet de faire de la lemmatisation en anglais, c'est-à-dire de réduire les mots à leur lemme (par exemple, "running" sera réduit à "run")

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer # Ces classes permettent de transformer le texte du roman en une matrice de caractéristiques (features) pour faire de l'apprentissage automatique. CountVectorizer compte le nombre d'occurrences de chaque mot, tandis que TfidfVectorizer calcule le poids de chaque mot en fonction de sa fréquence dans le document et dans l'ensemble des documents.
from sklearn.metrics.pairwise import cosine_similarity # Cette fonction permet de calculer la similarité cosinus entre deux vecteurs de caractéristiques, ce qui peut être utilisé pour mesurer la similarité entre deux textes (par exemple, pour faire du clustering ou de la classification)

In [3]:
nltk.download('punkt')       # Tokeniseur de phrases et mots
nltk.download('punkt_tab')   # Ressource complémentaire (versions récentes de NLTK)
nltk.download('stopwords')   # Listes de mots vides (français, anglais...)
nltk.download('wordnet')     # Base lexicale pour la lemmatisation
nltk.download('omw-1.4')    # Open Multilingual Wordnet (multilingue)

[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1006)>
[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1006)>
[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1006)>
[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1006)>
[nltk_data] Error loading omw-1.4: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1006)>


False

# Verification des versions des librairies

In [4]:
print(nltk.__version__, spacy.__version__, sklearn.__version__, pd.__version__)

3.9.4 3.8.14 1.9.0 3.0.3


# Telechargement du roman

In [5]:
import requests # Cette librairie permet de faire des requêtes HTTP pour récupérer le texte du roman depuis un site web

URL = 'https://www.gutenberg.org/files/17989/17989-0.txt'

# Téléchargement
response = requests.get(URL)
response.encoding = 'utf-8'
raw_text = response.text

print(f'Téléchargement terminé.')
print(f'Taille brute : {len(raw_text):,} caractères')
print(f'Aperçu :\n{raw_text[:300]}')

Téléchargement terminé.
Taille brute : 727,600 caractères
Aperçu :
*** START OF THE PROJECT GUTENBERG EBOOK 17989 ***




LE COMTE DE MONTE-CRISTO

Alexandre Dumas

Tome I (1845-1846)




Table des matières


Alexandre Dumas
I  Marseille.--L'arrivée.
II  Le père et le fils.
III  Les Catalans.
IV  Complot.
V  Le repas des fiançailles.
VI  Le substitut du procureur d


# Extraction du contenu (sans les métadonnées Gutenberg)

In [6]:
# Le fichier Gutenberg contient un en-tête et un pied de page
# On extrait uniquement le roman
DEBUT = '*** START OF THE PROJECT GUTENBERG EBOOK 17989 ***'
FIN   = '*** END OF THE PROJECT GUTENBERG EBOOK 17989 ***'

roman = raw_text.split(DEBUT)[1].split(FIN)[0]

print(f'Taille du roman (sans métadonnées) : {len(roman):,} caractères')
print(f'Nombre de lignes : {roman.count(chr(10)):,}')
print(f'Aperçu :\n{roman[:500]}')

Taille du roman (sans métadonnées) : 727,501 caractères
Nombre de lignes : 16,694
Aperçu :





LE COMTE DE MONTE-CRISTO

Alexandre Dumas

Tome I (1845-1846)




Table des matières


Alexandre Dumas
I  Marseille.--L'arrivée.
II  Le père et le fils.
III  Les Catalans.
IV  Complot.
V  Le repas des fiançailles.
VI  Le substitut du procureur du roi.
VII  L'interrogatoire.
VIII  Le château d'If.
IX  Le soir des fiançailles.
X  Le petit cabinet des Tuileries.
XI  L'Ogre de Corse.
XII  Le père et le fils.
XIII  Les Cent-Jours.
XIV  Le prisonnier furieux et le prisonnier fou.
XV  Le numéro 34


# Sauvegarde locale (optionnel)

In [7]:
# Sauvegarder le roman en local pour éviter de re-télécharger
with open('monte_cristo.txt', 'w', encoding='utf-8') as f:
    f.write(roman)

# Chargement depuis le fichier local :
# with open('monte_cristo.txt', encoding='utf-8') as f:
#     roman = f.read()

# ✏️  Travail demandé

# a)  Téléchargez le roman et affichez :

In [8]:
#     - Le nombre total de caractères (len)
print(f'Taille du roman (sans métadonnées) : {len(roman):,} caractères')
#     - Le nombre total de lignes (count('\n'))
print(f'Nombre de lignes : {roman.count(chr(10)):,}')
#     - Les 500 premiers caractères
print(f'Aperçu :\n{roman[:500]}')
#     - Les 200 derniers caractères (pour voir la fin du fichier Gutenberg)
print(f'Fin :\n{roman[-200:]}')

Taille du roman (sans métadonnées) : 727,501 caractères
Nombre de lignes : 16,694
Aperçu :





LE COMTE DE MONTE-CRISTO

Alexandre Dumas

Tome I (1845-1846)




Table des matières


Alexandre Dumas
I  Marseille.--L'arrivée.
II  Le père et le fils.
III  Les Catalans.
IV  Complot.
V  Le repas des fiançailles.
VI  Le substitut du procureur du roi.
VII  L'interrogatoire.
VIII  Le château d'If.
IX  Le soir des fiançailles.
X  Le petit cabinet des Tuileries.
XI  L'Ogre de Corse.
XII  Le père et le fils.
XIII  Les Cent-Jours.
XIV  Le prisonnier furieux et le prisonnier fou.
XV  Le numéro 34
Fin :
donna sans réserve et finit
par retomber haletant, brûlé de fatigue, épuisé de volupté, sous les
baisers de ces maîtresses de marbre et sous les enchantements de ce rêve
inouï.

FIN DU TOME PREMIER





# b)  Comptez le nombre de mots en utilisant simplement .split()

In [9]:
#     (estimation rapide, avant toute tokenisation avancée)
print(f'Nombre de mots : {len(roman.split()):,}')

Nombre de mots : 121,579


# c)  Combien de fois apparaît le nom 'Dantès' dans le roman ?

In [10]:
#     Utilisez roman.lower().count('dantès')
print(f'Nombre d\'apparitions de Dantès : {roman.lower().count("dantès"):,}')

Nombre d'apparitions de Dantès : 777


# d)  Cherchez les 5 premières lignes contenant le mot 'vengeance'.

In [11]:
vengeance_lines = [line for line in roman.split('\n') if 'vengeance' in line]
for i, line in enumerate(vengeance_lines[:5]):
    print(f'Ligne {i+1} : {line}')

Ligne 1 : que Fernand surtout était terrible dans sa vengeance.»
Ligne 2 : «À la bonne heure, continua Danglars; ainsi votre vengeance aurait le
Ligne 3 : menaçant et fort pour toutes les vengeances; alors il manifesta à M.
Ligne 4 : qui, pour lui aussi, était devenu messager d'une rude vengeance. Alors,
Ligne 5 : Il se disait que c'était la haine des hommes et non la vengeance de Dieu


# e)  Estimez le nombre de pages du roman (supposer 300 mots / page).

In [12]:
estimated_pages = len(roman.split()) // 300
print(f'Estimation du nombre de pages : {estimated_pages:,}')

Estimation du nombre de pages : 405


# Test de notre package

In [13]:
from chargement_corpus.load import CorpusLoader

roman = CorpusLoader.load_corpus('https://www.gutenberg.org/files/17989/17989-0.txt')

# taille et aperçu du roman
print(f'Taille du roman (sans métadonnées) : {len(roman):,} caractères')
print(f'Nombre de lignes : {roman.count(chr(10)):,}')
print(f'Aperçu :\n{roman[:500]}')


Taille du roman (sans métadonnées) : 727,493 caractères
Nombre de lignes : 16,686
Aperçu :
LE COMTE DE MONTE-CRISTO

Alexandre Dumas

Tome I (1845-1846)




Table des matières


Alexandre Dumas
I  Marseille.--L'arrivée.
II  Le père et le fils.
III  Les Catalans.
IV  Complot.
V  Le repas des fiançailles.
VI  Le substitut du procureur du roi.
VII  L'interrogatoire.
VIII  Le château d'If.
IX  Le soir des fiançailles.
X  Le petit cabinet des Tuileries.
XI  L'Ogre de Corse.
XII  Le père et le fils.
XIII  Les Cent-Jours.
XIV  Le prisonnier furieux et le prisonnier fou.
XV  Le numéro 34 et l


# 🧹 NETTOYAGE

In [14]:
# Observez le texte brut avant nettoyage
print(repr(roman[1000:1200]))   # repr() montre les caractères cachés (\n, \t...)

"e la Garde signala le\ntrois-mâts le _Pharaon_, venant de Smyrne, Trieste et Naples.\n\nComme d'habitude, un pilote côtier partit aussitôt du port, rasa le\nchâteau d'If, et alla aborder le navire entre l"


In [15]:
def clean_text(text):
    """Nettoie un texte pour une analyse NLP en français."""
    # 1. Mise en minuscules
    text = text.lower()
    # 2. Suppression des marqueurs d'italique Gutenberg (_mot_)
    text = re.sub(r'_([^_]+)_', r'\1', text)
    # 3. Remplacement des sauts de ligne et tabulations par des espaces
    text = re.sub(r'[\n\r\t]+', ' ', text)
    # 4. Suppression de la ponctuation et des caractères spéciaux
    #    On conserve les lettres françaises accentuées
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ' ]", ' ', text)
    # 5. Normalisation des apostrophes typographiques
    text = re.sub(r"[''`]", "'", text)
    # 6. Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text)
    # 7. Suppression des espaces en début et fin
    text = text.strip()
    return text

roman_clean = clean_text(roman)

print(f'Taille avant : {len(roman):,} caractères')
print(f'Taille après : {len(roman_clean):,} caractères')
print(f'Réduction   : {(1 - len(roman_clean)/len(roman))*100:.1f}%')
print(f'\nAperçu nettoyé :\n{roman_clean[:400]}')

Taille avant : 727,493 caractères
Taille après : 695,422 caractères
Réduction   : 4.4%

Aperçu nettoyé :
le comte de monte cristo alexandre dumas tome i table des matières alexandre dumas i marseille l'arrivée ii le père et le fils iii les catalans iv complot v le repas des fiançailles vi le substitut du procureur du roi vii l'interrogatoire viii le château d'if ix le soir des fiançailles x le petit cabinet des tuileries xi l'ogre de corse xii le père et le fils xiii les cent jours xiv le prisonnier 


In [16]:
extrait = roman[2800:3100]
print('AVANT NETTOYAGE :')
print(extrait)
print('\nAPRÈS NETTOYAGE :')
print(clean_text(extrait))

AVANT NETTOYAGE :
se de la Réserve.

En voyant venir cet homme, le jeune marin quitta son poste à côté du
pilote, et vint, le chapeau à la main, s'appuyer à la muraille du
bâtiment.

C'était un jeune homme de dix-huit à vingt ans, grand, svelte, avec de
beaux yeux noirs et des cheveux d'ébène; il y avait dans toute s

APRÈS NETTOYAGE :
se de la réserve en voyant venir cet homme le jeune marin quitta son poste à côté du pilote et vint le chapeau à la main s'appuyer à la muraille du bâtiment c'était un jeune homme de dix huit à vingt ans grand svelte avec de beaux yeux noirs et des cheveux d'ébène il y avait dans toute s


In [17]:
# ✏️  Travail demandé
# a)  Appliquez clean_text sur le roman. Affichez la taille avant et après.
cleaned_roman = clean_text(roman)
print(f'Taille avant : {len(roman):,} caractères')
print(f'Taille après : {len(cleaned_roman):,} caractères')

Taille avant : 727,493 caractères
Taille après : 695,422 caractères


In [18]:
# b)  Testez votre fonction sur cet extrait bruité :
#     '«Ah! c\'est vous, Dantès!» cria l\'homme... (numéro 34)\n\n-- Il est mort!!'
#     Que reste-t-il après nettoyage ?
extrait_bruite = '«Ah! c\'est vous, Dantès!» cria l\'homme... (numéro 34)\n\n-- Il est mort!!'
cleaned_extrait = clean_text(extrait_bruite)
print(f'Extrait avant nettoyage : {extrait_bruite}')
print(f'Extrait après nettoyage : {cleaned_extrait}')

Extrait avant nettoyage : «Ah! c'est vous, Dantès!» cria l'homme... (numéro 34)

-- Il est mort!!
Extrait après nettoyage : ah c'est vous dantès cria l'homme numéro il est mort


In [19]:
# c)  Le nom 'Monte-Cristo' contient un tiret. Après nettoyage, devient-il 'monte cristo'
#     (2 mots) ou 'montécristo' ? Proposez une solution pour le conserver en un seul token.
extrait_monte_cristo = 'Monte-Cristo'
cleaned_monte_cristo = clean_text(extrait_monte_cristo)
print(f'Extrait avant nettoyage : {extrait_monte_cristo}')
print(f'Extrait après nettoyage : {cleaned_monte_cristo}')

Extrait avant nettoyage : Monte-Cristo
Extrait après nettoyage : monte cristo


In [20]:
# d)  Modifiez clean_text pour également supprimer les nombres isolés (ex: '24', '1815').
#     Hint : re.sub(r'\b\d+\b', ' ', text)
def clean_text(text):
    """Nettoie un texte pour une analyse NLP en français."""
    # 1. Mise en minuscules
    text = text.lower()
    # 2. Suppression des marqueurs d'italique Gutenberg (_mot_)
    text = re.sub(r'_([^_]+)_', r'\1', text)
    # 3. Remplacement des sauts de ligne et tabulations par des espaces
    text = re.sub(r'[\n\r\t]+', ' ', text)
    # 4. Suppression de la ponctuation et des caractères spéciaux
    #    On conserve les lettres françaises accentuées
    text = re.sub(r"[^a-zàâçéèêëîïôûùüÿñæœ' ]", ' ', text)
    # 5. Normalisation des apostrophes typographiques
    text = re.sub(r"[''`]", "'", text)
    # 6. Suppression des nombres isolés
    text = re.sub(r'\b\d+\b', ' ', text)
    # 7. Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text)
    # 8. Suppression des espaces en début et fin
    text = text.strip()
    return text

In [21]:
# e)  Écrivez une version clean_text_leger qui ne fait que la mise en minuscule et
#     la suppression des espaces multiples. Comparez les résultats.
def clean_text_leger(text):
    """Nettoie un texte pour une analyse NLP en français (version légère)."""
    # 1. Mise en minuscules
    text = text.lower()
    # 2. Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text)
    # 3. Suppression des espaces en début et fin
    text = text.strip()
    return text

In [24]:
from nettoyage_corpus.cleaning import TextCleaner

# exemple de texte à nettoyer
texte = '«Ah! c\'est vous, Dantès!» cria l\'homme... (numéro 34)\n\n-- Il est mort!! Monte-Cristo'

cleaner = TextCleaner()
cleaned_texte = cleaner.clean_text_V3(texte)
print(f'Extrait avant nettoyage : {texte}')
print(f'Extrait après nettoyage : {cleaned_texte}')

AttributeError: 'TextCleaner' object has no attribute 'lower'